In [ ]:
def parse_ckt_file(file_name):
  try:
    with open(file_name, 'r') as file:
      for line in fle.readlines():
        print(line)



In [ ]:
eq = []
for dic in components:
    if dic['type'] == 'Resistor':
        eq.append(f"(({node_map[dic['nodes'][0]]} - {node_map[dic['nodes'][1]]}) / {dic['name']}) ")
    if dic['type'] == "Voltage_source":
        eq.append(f"({node_map[dic['nodes'][0]]} - {node_map[dic['nodes'][1]]}) = {dic['name']}")
    if dic['type'] == "Current_Source":
        eq.append(f"{dic['name']}")
    # Remove the trailing '+' if it exists
                  #if eq.endswith(' + '):
    #eq = eq[:-3]

for each node we have to add?

print(eq)


In [ ]:
import numpy as np
def parse_ckt_file(file_name):
    components = []
    component_types = { 'V': 'Voltage_source', 'I': 'Current_source' , 'R': 'Resistor'}
    with open(file_name, 'r') as fle:
      in_ckt = False
      for line in fle.readlines():
        line = line.strip()
        line = line.split('#', 1)[0]
        line = line.rstrip()
        if line == '.circuit':
           in_ckt = True
           continue
        if line == '.end':
           in_ckt = False
           continue
        if in_ckt:
           parts = line.split()

        if not parts:
           continue
        components.append({
        "name" : parts[0],
        "type": component_types.get(parts[0][0], "This type of component is not Supported"),
        'nodes' : [parts[1],parts[2]],
        'value' : float(parts[-1]),
        'add_info' : parts[3:-1]
        })


    fle.close()
    return components

components = parse_ckt_file('test_1.ckt')

print(components)

node_list = []
for dic in components:
    for node in dic['nodes']:
        if node != 'GND' and node not in node_list:
            node_list.append(node)
node_list.append('GND')

num_nodes = len(node_list)- 1   #ignoring ground
node_map = {}
assign = 1
for i in node_list:
    if i == 'GND':
        node_map[i] = 0
    else:
      node_map[i] = assign
      assign += 1
print(node_map)


voltage_sources = [component for component in components if component['type'] == 'Voltage_source']
num_vsources = len(voltage_sources)
print(num_vsources)
A = np.zeros((num_nodes + num_vsources, num_nodes + num_vsources))
b = np.zeros(num_nodes + num_vsources)

print(num_nodes)
for component in components:
    node_indices = [node_map[i] for i in component['nodes']]
    value = component['value']
    print(node_indices, value)
    if component ['type'] == 'Resistor':
        n1, n2 = node_indices
        conduc = 1 / value
        A[n1, n1] += conduc
        A[n2, n2] += conduc
        A[n1, n2] -= conduc
        A[n2, n1] -= conduc

    elif component['type'] == 'Voltage_source':
        vs_no = num_nodes + voltage_sources.index(component)
        n1, n2 = node_indices
        print(vs_no, num_nodes)
        b[vs_no] = value
        A[n1, vs_no] = 1
        A[n2, vs_no] = -1
        A[vs_no, n1] = 1
        A[vs_no, n2] = -1

    elif component['type'] == 'Current_source':
        n1, n2 = node_indices
        b[n1] -= value
        b[n2] += value

In [ ]:

# Extract node voltages and currents through voltage sources
voltages = voltages_currents[:num_nodes]
currents = voltages_currents[num_nodes:]
print(voltages, currents)

V_dic = {}
idx = 0
for dic in components:
    if dic['type'] == "Resistor":
      for node in dic['nodes']:
          if node not in V_dic:
              V_dic[node] = voltages[idx]
              idx += 1
    V_dic['GND'] = 0
print(V_dic)

# Calculate the current through each voltage source
idx = 0
I_dic = {}
for component in components:
    if component['type'] == 'Voltage_source':
        I_dic[component['name']] = currents[idx]
        idx += 1

print(I_dic)